### Process Data, for 1DCNN prediction.  
Date: March 27, 2026 Friday 2 am start. 
Author: Jennifer Yoon  

**original data file format:**  
 * .npy only used
 * shape [15000, 4], labels (I, II, V2, V5)
 * All 4 files, 60 seconds exactly at 250 hertz
 * float 16, max value +1.0, min value -1.0, units millivolts

**modified data file format:**   
 * shape [4, 1250], labels (I, II, V2, V5).  
 * .mat format, struct dict with 'val' key, data is type ndarray.  
 * split each user file (60 seconds, 250 hertz) into 6 files (10 seconds, 125 hertz), of shape [4, 1250]
 * downsampled from 250 to 125 hertz.
 * Transpose rows & columns.
 * file names use U000_S000.mat format. User 0, file 0 to 6.

**other considerations:**
 * Peak centering will use channel II only. Channel 1 has to many noisy spikes.  
 * No noise filtering applied. Prediction should still work reasonably well.  
 * Later, try bandpass filter and baseline wonder fix (channel V2 has wonder problem).
 * Generally the signals are not centered on zero amplitute. Will this be a problem? Consider later.    


In [2]:
# imports, short list initially  
import numpy as np 
import pandas as pd
import scipy.io
print(scipy.__version__)  
# when prints version, scipy.io is in current env.

1.15.3


In [11]:
pathin = "./data_predict/"
pathout = "./data_mod_predict/"  
filein = "ecg_4canales_1.npy"
fileout = "U000_S000.mat"
# six out files per in file. User 000, Sample 000 to 005.  

print("input:", pathin + filein)
print("output:", pathout + fileout)

input: ./data_predict/ecg_4canales_1.npy
output: ./data_mod_predict/U000_S000.mat


In [7]:
# load first user recording. 
data = np.load(pathin + filein)
print(data, data.shape, type(data))

[[ 0.55770783 -0.56275764 -0.56965174  0.15444015]
 [ 0.57164869 -0.56574271 -0.58576167  0.19121162]
 [ 0.53613618 -0.62842928 -0.59950249  0.16602317]
 ...
 [ 0.47602814 -0.71191218 -0.54462873  0.06278527]
 [ 0.49735258 -0.69470405 -0.60124245 -0.0309362 ]
 [-0.7715239  -1.         -1.         -0.78496805]] (15000, 4) <class 'numpy.ndarray'>


In [8]:
data = data.T
print(data.shape)

(4, 15000)


In [9]:
data_dw = data[:, ::2]
print(data_dw, data_dw.shape)

[[ 0.55770783  0.53613618  0.49152542 ...  0.49430623  0.56132589
   0.49735258]
 [-0.56275764 -0.62842928 -0.61791045 ... -0.66829847 -0.6728972
  -0.69470405]
 [-0.56965174 -0.59950249 -0.6230751  ... -0.54462873 -0.52739846
  -0.60124245]
 [ 0.15444015  0.16602317  0.18735062 ...  0.07353687  0.14372654
  -0.0309362 ]] (4, 7500)


In [12]:
# Test spitting code on 1 file first. 
# spit into 6 files, each 10 seconds at 125Hz, shape (4, 1250).  
segment_length = 1250 
num_segments = 6
user = 0

# Split and save into .mat files
for i in range(num_segments):
    start = i * segment_length
    end = start + segment_length
    
    # Extract 10-second segment: shape (4, 1250)
    segment = data_dw[:, start:end]
    
    # Format filename: U000_S00[i].mat
    file_name = f"U00{user}_S00{i}.mat"
    
    # Save as dictionary with 'val' key for 1DCNN compatibility
    scipy.io.savemat((pathout + file_name), {'val': segment})
    
    print(f"Saved: {file_name} | Shape: {segment.shape}")

Saved: U000_S000.mat | Shape: (4, 1250)
Saved: U000_S001.mat | Shape: (4, 1250)
Saved: U000_S002.mat | Shape: (4, 1250)
Saved: U000_S003.mat | Shape: (4, 1250)
Saved: U000_S004.mat | Shape: (4, 1250)
Saved: U000_S005.mat | Shape: (4, 1250)


In [19]:
# test, open a saved .mat file  
# Load the .mat file
mat_data = scipy.io.loadmat((pathout + "U000_S005.mat"), struct_as_record=False, squeeze_me=True)
print(mat_data, type(mat_data))
# Good, verified first and last samples.  

{'__header__': b'MATLAB 5.0 MAT-file Platform: posix, Created on: Fri Mar 27 04:05:47 2026', '__version__': '1.0', '__globals__': [], 'val': array([[ 0.46584467,  0.54761905,  0.38846372, ...,  0.49430623,
         0.56132589,  0.49735258],
       [-0.53089888, -0.5505618 , -0.51123596, ..., -0.66829847,
        -0.6728972 , -0.69470405],
       [-0.57741879, -0.63993702, -0.61544293, ..., -0.54462873,
        -0.52739846, -0.60124245],
       [ 0.11632134, -0.00199927,  0.15448928, ...,  0.07353687,
         0.14372654, -0.0309362 ]], shape=(4, 1250))} <class 'dict'>


### Define function to loop for rest of input files.

In [20]:
# Repeat for other files.  
# Define function  
def split_and_save_ecg(fname, uid, pathin, pathout):
    # local constants  
    segment_length = 1250 
    num_segments = 6
    
    # 1. Load and Transpose
    # input file is numpy format.  
    data = np.load(pathin + fname)
    data = data.T  # (15000, 4) -> (4, 15000)
    
    # 2. Downsample (250Hz -> 125Hz)
    data_dw = data[:, ::2]
    
    # 3. Split into 10-second segments (1250 samples each)
    for i in range(num_segments):
        start = i * segment_length
        end = start + segment_length
        segment = data_dw[:, start:end]
        
        # Format: U00[user]_S00[sample].mat
        # Note, need to update when more than single digit user id and sample index.  
        fileout = f"U00{uid}_S00{i}.mat"
            
        scipy.io.savemat((pathout + fileout), {'val': segment})
    
    print(f"Processed User {uid}: {fname} -> {num_segments} segments saved.")


In [21]:
# List of files and corresponding User IDs
files_to_process = [
    ("ecg_4canales_2.npy", 1),
    ("ecg_4canales_3.npy", 2),
    ("ecg_4canales_4.npy", 3)]

#pathin = defined at top  
#pathout = defined at top

# Run the loop
for fname, uid in files_to_process:
    split_and_save_ecg(fname, uid, pathin=pathin, pathout=pathout)

Processed User 1: ecg_4canales_2.npy -> 6 segments saved.
Processed User 2: ecg_4canales_3.npy -> 6 segments saved.
Processed User 3: ecg_4canales_4.npy -> 6 segments saved.


### Peak Centering, create .csv file controller

In [22]:
import numpy as np
import scipy.io as sio
import scipy.signal as signal
import pandas as pd
import os

In [26]:
# Test on single file first.  
# define function to find peaks, uses lead II.  
def process_single_file_to_csv(mat_file, path="./data_mod_predict/"):
    # 1. Parse IDs from filename (e.g., "U001_S002.mat")
    base_name = os.path.basename(mat_file)
    parts = base_name.replace('.mat', '').split('_')
    user_idx = parts[0].replace('U', '')
    sample_idx = parts[1].replace('S', '')
    
    # 2. Load the .mat file
    mat_data = sio.loadmat(os.path.join(path, mat_file))
    val = mat_data['val']  # Shape (4, 1250)
    
    # 3. Peak detection on Channel II (Index 1)
    # Using scipy to find R-peaks; adjust 'distance' based on 125Hz (e.g., ~0.6s apart)
    channel_ii = val[1, :]
    peaks, _ = signal.find_peaks(channel_ii, distance=75, height=np.mean(channel_ii))
    
    # 4. Define Leads
    leads = ["I", "II", "V2", "V5"]
    rows = []
    
    # 5. Build CSV records for each peak and each lead
    for win_idx, peak_pos in enumerate(peaks):
        for lead_idx, lead_name in enumerate(leads):
            rows.append({
                "filename": base_name,
                "user_idx": user_idx,
                "sample_idx": sample_idx,
                "lead_name": lead_name,
                "window_idx": win_idx,
                "peak_position": peak_pos
            })
            
    return rows

# --- Execution ---
# Load a single file as requested
target_file = "U000_S003.mat" 
all_records = process_single_file_to_csv(target_file)

# Create and save the CSV
df = pd.DataFrame(all_records)
df.to_csv("cnn_ready_predict_v1.csv", index=False)

# Divide by 4 because each peak created 4 rows (one for each lead)
num_peaks = len(all_records) // 4
print(f"CSV saved with {num_peaks} heart-beats detected in {target_file}.")

CSV saved with 14 heart-beats detected in U000_S003.mat.


In [ ]:
# saved python file, "peak_center_batch.py"  
# Fixed sized window, 250 data points, 125 on each side.  
# Zero padded? 
# I think this was the correct window size. Check with previous code.
# March 27, 2026 Friday 6am stoppped. 